<a href="https://www.kaggle.com/code/neharaj1518/s5e9-predicting-the-beats-per-minute-of-songs?scriptVersionId=357950915" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

import pandas as pd

# Load training dataset
train_path = "/kaggle/input/playground-series-s5e9/train.csv"
train_df = pd.read_csv(train_path)
# Load test dataset
test_path = "/kaggle/input/playground-series-s5e9/test.csv"
test_df = pd.read_csv(test_path)

# Features and target from training data
X = train_df.drop(columns=["id", "BeatsPerMinute"])
y = train_df["BeatsPerMinute"]

# Train/validation split to check baseline performance
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [2]:
# Load the training dataset
train_path = "/kaggle/input/playground-series-s5e9/train.csv"
train_df = pd.read_csv(train_path)

# Show basic info: shape, columns, first few rows
train_shape = train_df.shape
train_columns = train_df.columns.tolist()
train_head = train_df.head()

train_shape, train_columns, train_head

((524164, 11),
 ['id',
  'RhythmScore',
  'AudioLoudness',
  'VocalContent',
  'AcousticQuality',
  'InstrumentalScore',
  'LivePerformanceLikelihood',
  'MoodScore',
  'TrackDurationMs',
  'Energy',
  'BeatsPerMinute'],
    id  RhythmScore  AudioLoudness  VocalContent  AcousticQuality  \
 0   0     0.603610      -7.636942      0.023500         0.000005   
 1   1     0.639451     -16.267598      0.071520         0.444929   
 2   2     0.514538     -15.953575      0.110715         0.173699   
 3   3     0.734463      -1.357000      0.052965         0.001651   
 4   4     0.532968     -13.056437      0.023500         0.068687   
 
    InstrumentalScore  LivePerformanceLikelihood  MoodScore  TrackDurationMs  \
 0           0.000001                   0.051385   0.409866      290715.6450   
 1           0.349414                   0.170522   0.651010      164519.5174   
 2           0.453814                   0.029576   0.423865      174495.5667   
 3           0.159717                   0.0

In [3]:
from sklearn.model_selection import GridSearchCV

In [4]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None]
}

print(param_grid)

{'n_estimators': [100, 200, 300], 'max_depth': [10, 20, None]}


In [5]:
# Reduce the size of the training data for GridSearchCV to avoid memory issues
X_train_subset, _, y_train_subset, _ = train_test_split(X_train, y_train, train_size=0.05, random_state=42)

rf = RandomForestRegressor(n_estimators=50, random_state=42) # Reduced n_estimators
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1)
grid_search.fit(X_train_subset, y_train_subset)

best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("Best Hyperparameters:", best_params)
print("Best Score:", best_score)

Best Hyperparameters: {'max_depth': 10, 'n_estimators': 300}
Best Score: -0.004298853800751419


In [6]:
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# Train a new model with the best hyperparameters
best_rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
best_rf.fit(X_train, y_train)

# Validate the best model
y_val_pred_tuned = best_rf.predict(X_val)
# Calculate Mean Squared Error
mse_tuned = mean_squared_error(y_val, y_val_pred_tuned)
# Calculate RMSE
rmse_tuned = np.sqrt(mse_tuned)


print("RMSE with tuned hyperparameters:", rmse_tuned)

RMSE with tuned hyperparameters: 26.439261714308916


In [7]:
# Predict on test set
X_test = test_df.drop(columns=["id"])
test_preds_tuned = best_rf.predict(X_test)

print("Predictions on test set (first 5):", test_preds_tuned[:5])

Predictions on test set (first 5): [119.01586161 118.99759139 119.36864716 119.28524792 119.32877474]


In [8]:
# Create a DataFrame with IDs and predictions
submission_df = pd.DataFrame({'id': test_df['id'], 'BeatsPerMinute': test_preds_tuned})

# Save the submission DataFrame to a CSV file
submission_df.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' created successfully!")

Submission file 'submission.csv' created successfully!


In [9]:
# Create a DataFrame with IDs and predictions
submission_df = pd.DataFrame({'id': test_df['id'], 'BeatsPerMinute': test_preds_tuned})

# Display the first few rows of the submission DataFrame
print("Submission DataFrame (first 5):")
display(submission_df.head())

Submission DataFrame (first 5):


,id,BeatsPerMinute
0,524164,119.015862
1,524165,118.997591
2,524166,119.368647
3,524167,119.285248
4,524168,119.328775
